# Quantum Algorithms with Quantumium

This notebook demonstrates three landmark quantum algorithms:

1. **Deutsch-Jozsa** — Determine if a function is constant or balanced (exponential speedup)
2. **Grover's Search** — Find a marked item in an unsorted database (quadratic speedup)
3. **VQE** — Find the ground state energy of a quantum system (variational/hybrid)

In [ ]:
import numpy as np
from quantumium.algorithms.deutsch_jozsa import classify_function
from quantumium.algorithms.grover import grover_search
from quantumium.algorithms.vqe import vqe
from quantumium.algorithms.qaoa import qaoa_maxcut

## 1. Deutsch-Jozsa Algorithm

**Problem:** Given a black-box function f: {0,1}^n → {0,1}, determine whether f is:
- **Constant** — f(x) = 0 for all x, or f(x) = 1 for all x
- **Balanced** — f(x) = 0 for exactly half of inputs, f(x) = 1 for the other half

**Classical:** Need up to 2^(n-1) + 1 queries in the worst case.  
**Quantum:** Just **1 query** — an exponential speedup!

In [ ]:
n = 4  # 4-bit input → 16 possible inputs

# Constant function: always returns 0
result = classify_function(n, lambda x: 0)
print(f"f(x) = 0 for all x    → {result}")

# Constant function: always returns 1
result = classify_function(n, lambda x: 1)
print(f"f(x) = 1 for all x    → {result}")

# Balanced function: returns x mod 2
result = classify_function(n, lambda x: x % 2)
print(f"f(x) = x mod 2        → {result}")

# Balanced function: parity of x
result = classify_function(n, lambda x: bin(x).count('1') % 2)
print(f"f(x) = parity(x)      → {result}")

## 2. Grover's Search Algorithm

**Problem:** Find a marked item in an unsorted database of N items.

**Classical:** O(N) queries needed.  
**Quantum:** O(√N) queries — a quadratic speedup.

For a database of 1 million items, classical search needs ~1,000,000 lookups while
Grover's needs only ~1,000.

In [ ]:
# Search for item 11 in a 4-qubit database (16 items)
num_qubits = 4
target = 11

print(f"Searching for |{target}> among {2**num_qubits} items ({num_qubits} qubits)")
print(f"Target in binary: |{target:0{num_qubits}b}>\n")

qc = grover_search(num_qubits, [target])
counts = qc.measure_all(shots=1000)

print("Measurement results (top 5):")
for bitstring, count in sorted(counts.items(), key=lambda x: -x[1])[:5]:
    marker = " ← TARGET" if int(bitstring, 2) == target else ""
    print(f"  |{bitstring}> = {int(bitstring, 2):>2d}  ({count/10:.1f}%){marker}")

In [ ]:
# Search for multiple targets
targets = [3, 7]
print(f"Searching for |{targets[0]}> and |{targets[1]}> among {2**num_qubits} items\n")

qc = grover_search(num_qubits, targets)
counts = qc.measure_all(shots=1000)

print("Measurement results (top 5):")
for bitstring, count in sorted(counts.items(), key=lambda x: -x[1])[:5]:
    marker = " ← TARGET" if int(bitstring, 2) in targets else ""
    print(f"  |{bitstring}> = {int(bitstring, 2):>2d}  ({count/10:.1f}%){marker}")

## 3. Variational Quantum Eigensolver (VQE)

VQE is a **hybrid quantum-classical algorithm** for finding the ground state energy
of a Hamiltonian. It's one of the most promising algorithms for near-term quantum devices.

**How it works:**
1. Prepare a parameterized quantum circuit (ansatz)
2. Measure the energy expectation value
3. Use a classical optimizer to adjust parameters
4. Repeat until convergence

We'll find the ground state energy of a 2-qubit Heisenberg model:

$$H = X_0 X_1 + Y_0 Y_1 + Z_0 Z_1$$

In [ ]:
# Define Pauli matrices
I = np.eye(2)
X_mat = np.array([[0, 1], [1, 0]])
Y_mat = np.array([[0, -1j], [1j, 0]])
Z_mat = np.array([[1, 0], [0, -1]])

# 2-qubit Heisenberg Hamiltonian: H = XX + YY + ZZ
H_heisenberg = (
    np.kron(X_mat, X_mat) +
    np.kron(Y_mat, Y_mat) +
    np.kron(Z_mat, Z_mat)
)

# Exact ground state energy (from diagonalization)
eigenvalues = np.linalg.eigvalsh(H_heisenberg)
exact_energy = eigenvalues[0]
print(f"Exact ground state energy: {exact_energy:.6f}")
print(f"All eigenvalues: {np.round(eigenvalues, 4)}")

In [ ]:
# Run VQE
result = vqe(
    num_qubits=2,
    hamiltonian=H_heisenberg,
    ansatz_depth=2,
    max_iter=200,
    num_starts=5,
)

print(f"VQE result:")
print(f"  Optimal energy: {result.optimal_energy:.6f}")
print(f"  Exact energy:   {exact_energy:.6f}")
print(f"  Error:          {abs(result.optimal_energy - exact_energy):.6f}")
print(f"  Iterations:     {result.num_iterations}")

In [ ]:
# Plot the convergence history
try:
    import matplotlib.pyplot as plt
    plt.figure(figsize=(8, 4))
    plt.plot(result.history, 'b-', alpha=0.7, label='VQE energy')
    plt.axhline(y=exact_energy, color='r', linestyle='--', label=f'Exact ({exact_energy:.2f})')
    plt.xlabel('Iteration')
    plt.ylabel('Energy')
    plt.title('VQE Convergence')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
except ImportError:
    print("(matplotlib not available — install it to see convergence plots)")
    print(f"Energy history (first 10): {[round(e, 4) for e in result.history[:10]]}")
    print(f"Energy history (last 5):   {[round(e, 4) for e in result.history[-5:]]}")

## 4. QAOA for MaxCut

The **Quantum Approximate Optimization Algorithm (QAOA)** tackles combinatorial
optimization problems. Here we solve **MaxCut**: partition a graph's vertices into two
sets to maximize the number of edges between them.

This is NP-hard in general, but QAOA can find good approximate solutions.

In [ ]:
# Define a triangle graph: 3 vertices, 3 edges
#   0
#  / \
# 1---2
edges = [(0, 1), (0, 2), (1, 2)]
num_qubits = 3

result = qaoa_maxcut(
    num_qubits=num_qubits,
    edges=edges,
    p=2,              # 2 QAOA layers
    num_starts=10,
)

print(f"QAOA MaxCut result:")
print(f"  Best partition:  {result.optimal_bitstring}")
print(f"  Expected cut:    {result.optimal_value:.4f}")
print()

# Interpret the result
bs = result.optimal_bitstring
set_a = [i for i, b in enumerate(bs) if b == '0']
set_b = [i for i, b in enumerate(bs) if b == '1']
cut_edges = [(u, v) for u, v in edges if bs[u] != bs[v]]
print(f"  Set A: {set_a}")
print(f"  Set B: {set_b}")
print(f"  Cut edges: {cut_edges} ({len(cut_edges)} edges cut)")

In [ ]:
# Larger example: 4-vertex graph
#  0 - 1
#  |   |
#  3 - 2
edges_4 = [(0, 1), (1, 2), (2, 3), (3, 0)]

result_4 = qaoa_maxcut(
    num_qubits=4,
    edges=edges_4,
    p=2,
    num_starts=20,
)

print(f"4-vertex cycle MaxCut:")
print(f"  Best partition:  {result_4.optimal_bitstring}")
print(f"  Expected cut:    {result_4.optimal_value:.4f}")
bs4 = result_4.optimal_bitstring
cut_4 = [(u, v) for u, v in edges_4 if bs4[u] != bs4[v]]
print(f"  Edges cut: {len(cut_4)} / {len(edges_4)}")
print(f"  (Optimal for a 4-cycle is 4 — alternating partition)")

## Summary

| Algorithm | Problem | Speedup | Type |
|-----------|---------|---------|------|
| **Deutsch-Jozsa** | Function classification | Exponential | Exact |
| **Grover** | Unstructured search | Quadratic | Exact |
| **VQE** | Ground state energy | Problem-dependent | Variational |
| **QAOA** | Combinatorial optimization | Problem-dependent | Variational |

Quantumium also includes Shor's factoring, Simon's periodicity, HHL linear systems,
QPE phase estimation, QFT, and more. See the full API documentation for details.

Next: explore [quantum finance](quantum-finance.ipynb) for real-world applications.